# RAG Chain with Chroma Retriever

Chroma 벡터스토어를 다시 연결해 retriever 기반 RAG 체인을 구성하는 노트북


> 
> `질문 라우팅 -> source별 retriever -> context 압축 -> prompt -> answer`
>

### 환경 준비


In [3]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

current_dir = os.getcwd()
target_dir = os.path.join(current_dir, '../src')
sys.path.append(os.path.abspath(target_dir))

from rag_chain_utils import build_context, classify_question, make_search_kwargs

load_dotenv('.env')
load_dotenv('../.env')

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY를 찾지 못했습니다. .env 파일을 확인해 주세요.')


d:\encore\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 모델과 임베딩 준비


In [4]:
model = ChatOpenAI(model='gpt-4o-mini', temperature=0, timeout=60)

embedding_function = HuggingFaceEmbeddings(
    model_name='jhgan/ko-sroberta-multitask',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

print(type(model).__name__)
print(type(embedding_function).__name__)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7275.62it/s]


ChatOpenAI
HuggingFaceEmbeddings


### Chroma 벡터스토어 재연결


In [5]:
PERSIST_DIRECTORY = Path('../chroma_db')
COLLECTION_NAME = 'maplestory_guides'

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=str(PERSIST_DIRECTORY),
    embedding_function=embedding_function,
)

def get_retriever(question: str):
    route = classify_question(question)
    search_kwargs = make_search_kwargs(route)
    retriever = vectorstore.as_retriever(
        search_type='similarity',
        search_kwargs=search_kwargs,
    )
    return route, search_kwargs, retriever

print(type(vectorstore).__name__)
print('질문 라우팅 유틸 준비 완료')


Chroma
질문 라우팅 유틸 준비 완료


### Retriever 동작 확인


In [6]:
question = '메이플스토리 초보자에게 추천할 직업은 무엇인가요?'
route, search_kwargs, retriever = get_retriever(question)
docs = retriever.invoke(question)

print(f'route = {route}')
print(f'search_kwargs = {search_kwargs}')
print(f'검색 문서 수: {len(docs)}')
for i, doc in enumerate(docs[:3], start=1):
    print(f'[{i}] metadata = {doc.metadata}')
    print(doc.page_content[:220])
    print('-' * 80)


route = jobs
search_kwargs = {'k': 6, 'filter': {'source': 'jobs'}}
검색 문서 수: 6
[1] metadata = {'source': 'jobs', 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/35', 'job_id': '35', 'category': '도적', 'name': '팬텀'}
직업명: 팬텀
별명: 영웅이 된 괴도
설명: 소중한 사람을 잃은 후 메이플 월드를 지키기 위해 영웅이 된 괴도입니다. 높은 기동력으로 빠르게 이동하고 모험가 종족의 스킬을 훔쳐서 상황에 맞게 사용할 수 있습니다.
주스탯: LUK (행운)
무기: 케인 (전용무기)
--------------------------------------------------------------------------------
[2] metadata = {'name': '듀얼블레이드', 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/32', 'source': 'jobs', 'job_id': '32', 'category': '도적'}
직업명: 듀얼블레이드
별명: 숨겨진 각인
설명: 비밀 도적 조직인 비화원의 일원입니다. 메이플 월드의 도적들과의 오해를 풀고 행동을 함께합니다. 단검과 블레이드로 스타일리시한 연계기를 사용하며, 매우 빠른 속도로 강격한 공격을 연발합니다.
주스탯: LUK (행운)
무기: 단검, 듀얼블레이드 (전용 보조 무기)
--------------------------------------------------------------------------------
[3] metadata = {'url': 'https://maplestory.nexon.com/Guide/N23Job/View/12', 'job_id': '12', 'name': '제로', 'source': 'jobs', 'category': '전사'}
직업명: 제로


### Context 확인


In [7]:
context = build_context(docs)
print(context[:1200])


[문서 1]
source: jobs
title: 팬텀
url: https://maplestory.nexon.com/Guide/N23Job/View/35
excerpt: 직업명: 팬텀 별명: 영웅이 된 괴도 설명: 소중한 사람을 잃은 후 메이플 월드를 지키기 위해 영웅이 된 괴도입니다. 높은 기동력으로 빠르게 이동하고 모험가 종족의 스킬을 훔쳐서 상황에 맞게 사용할 수 있습니다. 주스탯: LUK (행운) 무기: 케인 (전용무기)

[문서 2]
source: jobs
title: 듀얼블레이드
url: https://maplestory.nexon.com/Guide/N23Job/View/32
excerpt: 직업명: 듀얼블레이드 별명: 숨겨진 각인 설명: 비밀 도적 조직인 비화원의 일원입니다. 메이플 월드의 도적들과의 오해를 풀고 행동을 함께합니다. 단검과 블레이드로 스타일리시한 연계기를 사용하며, 매우 빠른 속도로 강격한 공격을 연발합니다. 주스탯: LUK (행운) 무기: 단검, 듀얼블레이드 (전용 보조 무기)

[문서 3]
source: jobs
title: 제로
url: https://maplestory.nexon.com/Guide/N23Job/View/12
excerpt: 직업명: 제로 별명: 신의 아이 설명: 시간의 여신 륀느가 운명을 바꾸기 위해 탄생시킨 신의 아이입니다. 두 캐릭터를 교체해 공격할 수 있으며, 스킬을 연계할수록 화려하고 강력한 공격을 구사합니다. 주스탯: STR (힘) 무기: 알파 : 태도 (전용무기)베타 : 대검 (전용무기)

[문서 4]
source: jobs
title: 일리움
url: https://maplestory.nexon.com/Guide/N23Job/View/20
excerpt: 직업명: 일리움 별명: 광휘의 날개 설명: 고대 크리스탈에게 선택받아 우든레프의 미래를 짊어진 마법사입니다. 마법과 고대 크리스탈을 공명시켜 글로리윙 모드에 진입해 자유롭게 비행하거나, 강력한 기계 소환수를 불러낼 수 있습니다. 주스탯: INT (

### RAG Prompt


In [8]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            'system',
            (
                '당신은 메이플스토리 정보를 안내하는 도우미입니다. '
                '반드시 제공된 context에 근거해서만 답변하세요. '
                '답변은 먼저 핵심 결론을 2~4문장으로 간결하게 제시하고, 이어서 근거가 된 문서 종류와 이유를 짧게 설명하세요. '
                '추천 질문이면 비교 기준을 분명히 밝히고, context가 부족하면 부족한 정보를 명확히 말하세요.'
            ),
        ),
        (
            'human',
            '[Context]\n{context}\n\n[Question]\n{question}',
        ),
    ]
)


### RAG Chain


In [9]:
def retrieve(question: str):
    _, _, retriever = get_retriever(question)
    return retriever.invoke(question)

rag_chain = (
    {
        'context': RunnableLambda(retrieve) | RunnableLambda(build_context),
        'question': RunnablePassthrough(),
    }
    | prompt | model | StrOutputParser()
)

def rag_answer(question: str) -> str:
    return rag_chain.invoke(question)


### 질문 & 답변 확인


In [11]:
question = input('질문을 입력해 주세요: ')
answer = rag_answer(question)
print(f'질문 : {question} \n답변 : {answer}')


질문 : 초보에게 좋은 무기는? 
답변 : 초보에게 좋은 무기는 차크람을 사용하는 칼리 직업의 무기입니다. 칼리는 공격과 이동 스킬을 연계하여 자유로운 전투를 펼칠 수 있어 초보자에게 적합합니다. 

이 정보는 문서 5에서 제공된 칼리 직업에 대한 설명을 기반으로 하였습니다. 칼리는 하이레프 종족의 사제로, 차크람을 사용하여 강력한 공격을 구사할 수 있습니다.
